In [4]:
import numpy as np
import pandas as pd
from skeLCS import eLCS
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

# Read dataset (turns off default behavior reading "none" as missing)
df = pd.read_csv("medical_insurance_cleaned.csv", keep_default_na=False, na_values=[""])

# Create 3 risk band target classes from continuous risk_score
df["risk_band"] = pd.qcut(df["risk_score"], 3, labels=False)
y = df["risk_band"].values

# Define leakage columns to drop alongside target and ID columns
LEAKAGE_COLS = [
    "is_high_risk",
    "annual_premium",
    "monthly_premium",
    "annual_medical_cost",
    "claims_count",
    "avg_claim_amount",
    "total_claims_paid",
]

# Baseline Feature Matrix (45 features)
X_baseline_df = df.drop(columns=LEAKAGE_COLS + ["risk_score", "risk_band", "person_id"])

# Categorical integer encoding for baseline features
for col in X_baseline_df.select_dtypes(exclude="number"):
    X_baseline_df[col] = X_baseline_df[col].astype("category").cat.codes

X_baseline = X_baseline_df.values.astype(float)

# Shared 3-Fold Stratified CV
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)



In [5]:
# ==============================================================================
# SECTION 2: MODEL 1 - BASELINE ORIGINAL eLCS (45 FEATURES)
# ==============================================================================
print("--- MODEL 1: BASELINE ORIGINAL eLCS ---")
model_baseline = eLCS(random_state=42)
baseline_scores = cross_val_score(
    model_baseline, X_baseline, y, cv=cv, scoring="balanced_accuracy"
)

print(f"Baseline eLCS (45 features) CV Balanced Accuracy: {baseline_scores.mean():.3f}")
print(f"Fold Scores: {np.round(baseline_scores, 3)}\n")

model_baseline.fit(X_baseline, y)
m1_coverage = model_baseline.get_final_instance_coverage()

--- MODEL 1: BASELINE ORIGINAL eLCS ---
Baseline eLCS (45 features) CV Balanced Accuracy: 0.568
Fold Scores: [0.505 0.641 0.56 ]



In [6]:
# ==============================================================================
# SECTION 3: MODEL 2 - DECISION TREE REFERENCE & FEATURE SELECTION
# ==============================================================================
print("--- MODEL 2: DECISION TREE REFERENCE ---")
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_scores = cross_val_score(
    tree, X_baseline, y, cv=cv, scoring="balanced_accuracy"
)

print(f"Decision Tree CV Balanced Accuracy: {tree_scores.mean():.3f}")

# Extract top features using decision tree feature importance
tree.fit(X_baseline, y)
feature_importances = pd.Series(tree.feature_importances_, index=X_baseline_df.columns)
top_features = feature_importances.sort_values(ascending=False).head(4).index.tolist()

print("\nTop 4 Selected Features:")
print(feature_importances.sort_values(ascending=False).head(4))
print()

--- MODEL 2: DECISION TREE REFERENCE ---
Decision Tree CV Balanced Accuracy: 0.953

Top 4 Selected Features:
chronic_count    0.404078
age              0.366792
smoker           0.160757
bmi              0.068372
dtype: float64



In [7]:
# ==============================================================================
# SECTION 4: MODEL 3 - 4-FEATURE SELECTED eLCS (DEFAULT HYPERPARAMETERS)
# ==============================================================================
print("--- MODEL 3: 4-FEATURE SELECTED eLCS ---")
X_selected_df = df[top_features].copy()

for col in X_selected_df.select_dtypes(exclude="number"):
    X_selected_df[col] = X_selected_df[col].astype("category").cat.codes

X_selected = X_selected_df.values.astype(float)

model_selected = eLCS(random_state=42)
selected_scores = cross_val_score(
    model_selected, X_selected, y, cv=cv, scoring="balanced_accuracy"
)

model_selected.fit(X_selected, y)
m3_coverage = model_selected.get_final_instance_coverage()

print(f"eLCS (4 features) CV Balanced Accuracy: {selected_scores.mean():.3f}")
print(f"Fold Scores: {np.round(selected_scores, 3)}\n")




--- MODEL 3: 4-FEATURE SELECTED eLCS ---
eLCS (4 features) CV Balanced Accuracy: 0.827
Fold Scores: [0.821 0.862 0.799]



In [8]:
# ==============================================================================
# SECTION 5: MODEL 4 - TUNED & COMPACTED eLCS
# ==============================================================================
print("--- MODEL 4: TUNED & COMPACTED eLCS ---")

# Step 1: Preprocessing with MinMaxScaler (Uniform predicate interval evolution)
scaler = MinMaxScaler()
X_tuned_scaled = scaler.fit_transform(X_selected)

# Step 2: Tuned Model Configuration (Subsumption + Extended Iterations + N)
model_tuned = eLCS(
    N=1500,
    learning_iterations=20000,
    nu=5,
    p_spec=0.5,
    do_GA_subsumption=True,
    do_correct_set_subsumption=True,
    random_state=42
)

tuned_scores = cross_val_score(
    model_tuned, X_tuned_scaled, y, cv=cv, scoring="balanced_accuracy"
)

print(f"Tuned & Compacted eLCS CV Balanced Accuracy: {tuned_scores.mean():.3f}")
print(f"Fold Scores: {np.round(tuned_scores, 3)}")

# Fit full model to evaluate coverage and rule population
model_tuned.fit(X_tuned_scaled, y)
m4_coverage = model_tuned.get_final_instance_coverage()
print(f"Final Training Accuracy: {model_tuned.get_final_accuracy():.3f}")
print(f"Final Instance Coverage: {model_tuned.get_final_instance_coverage():.1%}")

# Rule Compaction Pruning
raw_pop = model_tuned.population.popSet
pruned_rules = [r for r in raw_pop if r.fitness >= 0.50]
print(f"Raw Population Size: {len(raw_pop)} | Compacted/Pruned Rules (Fitness >= 0.5): {len(pruned_rules)}\n")



--- MODEL 4: TUNED & COMPACTED eLCS ---
Tuned & Compacted eLCS CV Balanced Accuracy: 0.860
Fold Scores: [0.841 0.856 0.882]
Final Training Accuracy: 0.843
Final Instance Coverage: 100.0%
Raw Population Size: 571 | Compacted/Pruned Rules (Fitness >= 0.5): 186



In [9]:
# ==============================================================================
# CONVENTIONAL MACHINE LEARNING MODELS
# ==============================================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression


print("--- CONVENTIONAL MODELS EVALUATION ---")

# 1. Decision Tree (Already Implemented)
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
cv_scores_dt = cross_val_score(tree, X_baseline, y, cv=cv, scoring="balanced_accuracy")

# 2. Random Forest Classifier (Ensemble Benchmark)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
cv_scores_rf = cross_val_score(rf, X_baseline, y, cv=cv, scoring="balanced_accuracy")

# 3. Logistic Regression (Linear Benchmark - using scaled features to ensure convergence)
scaler_conv = MinMaxScaler()
X_baseline_scaled = scaler_conv.fit_transform(X_baseline)

lr = LogisticRegression(max_iter=1000, random_state=42)
cv_scores_lr = cross_val_score(lr, X_baseline_scaled, y, cv=cv, scoring="balanced_accuracy")


print(f"Decision Tree CV Balanced Accuracy:      {cv_scores_dt.mean():.3f}")
print(f"Random Forest CV Balanced Accuracy:      {cv_scores_rf.mean():.3f}")
print(f"Logistic Regression CV Balanced Accuracy:{cv_scores_lr.mean():.3f}")

--- CONVENTIONAL MODELS EVALUATION ---
Decision Tree CV Balanced Accuracy:      0.953
Random Forest CV Balanced Accuracy:      0.841
Logistic Regression CV Balanced Accuracy:0.884


In [10]:
# ==============================================================================
# SECTION 6: MODEL 5 - FINAL COMBINED MODEL COMPARISON
# ==============================================================================
print("--- MODEL 5: FINAL COMBINED MODEL EVALUATION ---")

results_summary = pd.DataFrame({
    "Model Order": [1, 2, 3, 4],
    "Model Description": [
        "Baseline Original eLCS (45 features)",
        "Decision Tree Benchmark (Reference)",
        "4-Feature Selected eLCS (Default Params)",
        "Tuned & Compacted eLCS (Scaling + Subsumption + 20k Iters)"
    ],
    "CV Balanced Accuracy": [
        round(baseline_scores.mean(), 3),
        round(tree_scores.mean(), 3),
        round(selected_scores.mean(), 3),
        round(tuned_scores.mean(), 3)
    ]
   
})

display(results_summary)

--- MODEL 5: FINAL COMBINED MODEL EVALUATION ---


,Model Order,Model Description,CV Balanced Accuracy
0,1,Baseline Original eLCS (45 features),0.568
1,2,Decision Tree Benchmark (Reference),0.953
2,3,4-Feature Selected eLCS (Default Params),0.827
3,4,Tuned & Compacted eLCS (Scaling + Subsumption ...,0.860


In [11]:
# ==============================================================================
# INSTANT SUMMARY TABLE (Reuses existing notebook variables)
# ==============================================================================
results_summary = pd.DataFrame({
    "Category": [
        "LCS Model",
        "LCS Model",
        "LCS Model",
        "Conventional ML",
        "Conventional ML",
        "Conventional ML"
    ],
    "Model Name": [
        "Original eLCS (Raw/Minimally Processed, 45 Features)",
        "Original eLCS (4 Selected Features, Default Params)",
        "Tuned & Compacted eLCS (Scaling + Subsumption + 20k Iters)",
        "Decision Tree (Max Depth=5)",
        "Random Forest (100 Trees)",
        "Logistic Regression (Scaled)"
    ],
    "CV Balanced Accuracy": [
        round(baseline_scores.mean(), 3),
        round(selected_scores.mean(), 3),
        round(tuned_scores.mean(), 3),
        round(cv_scores_dt.mean(), 3),
        round(cv_scores_rf.mean(), 3),
        round(cv_scores_lr.mean(), 3)
    ],
    "Instance Coverage": [
        f"{m1_coverage:.1%}" if 'm1_coverage' in globals() else "N/A",
        f"{m3_coverage:.1%}" if 'm3_coverage' in globals() else "N/A",
        f"{m4_coverage:.1%}" if 'm4_coverage' in globals() else "N/A",
        "100.0%",
        "100.0%",
        "100.0%"
    ]
})

display(results_summary)

,Category,Model Name,CV Balanced Accuracy,Instance Coverage
0,LCS Model,"Original eLCS (Raw/Minimally Processed, 45 Fea...",0.568,74.5%
1,LCS Model,"Original eLCS (4 Selected Features, Default Pa...",0.827,100.0%
2,LCS Model,Tuned & Compacted eLCS (Scaling + Subsumption ...,0.860,100.0%
3,Conventional ML,Decision Tree (Max Depth=5),0.953,100.0%
4,Conventional ML,Random Forest (100 Trees),0.841,100.0%
5,Conventional ML,Logistic Regression (Scaled),0.884,100.0%


In [19]:
# ==============================================================================
# NEW CELL: DISPLAY TOP COMPACTED eLCS RULES FROM MODEL_TUNED
# ==============================================================================
import pandas as pd
import numpy as np

# 1. Grab feature names from X_selected used in Section 5
feature_names = list(X_selected.columns) if hasattr(X_selected, 'columns') else ["chronic_count", "age", "smoker", "bmi"]

# Map numeric target codes to human-readable risk labels
class_map = {0: "LOW RISK", 1: "MEDIUM RISK", 2: "HIGH RISK"}

rule_data = []

# 2. Extract rules from model_tuned population
for rule in model_tuned.population.popSet:
    # Filter for compacted/pruned rules (Fitness >= 0.50)
    if rule.fitness < 0.50:
        continue
        
    conditions = []
    for attr_idx, attribute in enumerate(rule.condition):
        # Skip wildcards ('#') or unspecified attributes
        if attribute is not None and attribute != '#':
            feat = feature_names[attr_idx]
            
            # Format numeric interval bounds [min, max] or discrete values
            if isinstance(attribute, (list, tuple, np.ndarray)):
                min_val, max_val = attribute[0], attribute[1]
                conditions.append(f"{min_val:.2f} <= {feat} <= {max_val:.2f}")
            else:
                conditions.append(f"{feat} == {attribute}")
    
    if_str = " AND ".join(conditions) if conditions else "IF [ANY PATIENT]"
    predicted_class = class_map.get(rule.phenotype, f"Class {rule.phenotype}")
    
    rule_data.append({
        "Rule Condition (IF)": if_str,
        "Predicted Risk Band (THEN)": predicted_class,
        "Fitness": round(rule.fitness, 3),
        "Accuracy": round(rule.accuracy, 3),
        #"Experience (Count)": rule.experience,
        "Numerosity": rule.numerosity
    })

# 3. Create DataFrame sorted by highest Fitness and Experience
df_top_rules = (
    pd.DataFrame(rule_data)
    .sort_values(by=["Fitness"], ascending=False)
    .reset_index(drop=True)
)

print(f"Displaying Top 10 Compacted Rules (out of {len(df_top_rules)} pruned rules):\n")
display(df_top_rules.head(10))

Displaying Top 10 Compacted Rules (out of 186 pruned rules):



,Rule Condition (IF),Predicted Risk Band (THEN),Fitness,Accuracy,Numerosity
0,-0.24 <= chronic_count <= 0.43 AND age == 1.0 ...,LOW RISK,1.0,1.0,15
1,0.04 <= chronic_count <= 0.43 AND age == 1.0 A...,LOW RISK,1.0,1.0,15
2,-0.33 <= chronic_count <= 0.43 AND age == 0.0 ...,LOW RISK,1.0,1.0,14
3,0.81 <= chronic_count <= 0.94,HIGH RISK,1.0,1.0,21
4,0.54 <= chronic_count <= 0.67 AND age == 0.0 A...,MEDIUM RISK,1.0,1.0,16
5,chronic_count == 0.16666666666666666 AND -0.15...,LOW RISK,1.0,1.0,16
6,chronic_count == 0.5 AND 0.38 <= age <= 0.90,HIGH RISK,1.0,1.0,15
7,0.56 <= chronic_count <= 0.88 AND age == 0.0,HIGH RISK,1.0,1.0,17
8,0.55 <= chronic_count <= 0.90 AND age == 0.5,HIGH RISK,1.0,1.0,12
9,chronic_count == 0.16666666666666666 AND -0.24...,LOW RISK,1.0,1.0,15
